In [ ]:
# Facility Location Notebook (alias-based, backend-agnostic test)

from optilab.aliases import Model, GRB, quicksum
import numpy as np

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()
print("Backend:", m.backend_name)

# -------------------------------------------------------------------------------
# Problem data
# -------------------------------------------------------------------------------
rng = np.random.default_rng(42)

# Number of facilities and clients
n_facilities = rng.integers(5, 15)
n_clients = rng.integers(20, 50)

# Assignment costs (client i -> facility j)
cost = rng.integers(10, 100, size=(n_clients, n_facilities))

# Fixed opening costs for each facility
fixed_cost = rng.integers(50, 200, size=n_facilities)

print("n facilities:", n_facilities)
print("n clients:", n_clients)
print("fixed costs:", fixed_cost)
print("cost matrix shape:", cost.shape)

# -------------------------------------------------------------------------------
# Decision variables
# -------------------------------------------------------------------------------
# y[j] = 1 if facility j is opened
y = [m.add_binary_var(name=f"y_{j}") for j in range(n_facilities)]

# x[i][j] = 1 if client i is assigned to facility j
x = [
    [m.add_binary_var(name=f"x_{i}_{j}") for j in range(n_facilities)]
    for i in range(n_clients)
]

# -------------------------------------------------------------------------------
# Constraints
# -------------------------------------------------------------------------------

# Each client must be assigned to exactly one facility
for i in range(n_clients):
    m.add_constraint(
        quicksum(x[i][j] for j in range(n_facilities)) == 1
    )

# Linking constraint: assignment only possible if facility is open
for i in range(n_clients):
    for j in range(n_facilities):
        m.add_constraint(
            x[i][j] <= y[j]
        )

# -------------------------------------------------------------------------------
# Objective
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(fixed_cost[j] * y[j] for j in range(n_facilities)) +
    quicksum(cost[i][j] * x[i][j]
             for i in range(n_clients)
             for j in range(n_facilities)),
    GRB.MINIMIZE
)

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract solution
# -------------------------------------------------------------------------------
open_facilities = np.array([v.x for v in y])
assignment = np.array([[v.x for v in row] for row in x])

print("\nOpen facilities:", open_facilities)

print("\nAssignments (client -> facility):")
for i in range(n_clients):
    j = np.argmax(assignment[i])
    print(f"Client {i:2d} -> Facility {j:2d}")

total_fixed = np.sum(fixed_cost * open_facilities)
total_assign = np.sum(cost * assignment)

print("\nTotal fixed cost:", float(total_fixed))
print("Total assignment cost:", float(total_assign))
print("Total cost:", float(total_fixed + total_assign))